# Variedade intra-identidade no CelebA

Pergunta que este notebook responde: **para quantas identidades existe mais de uma foto, e para quantas dessas o atributo (ex.: `Smiling`) realmente varia entre as fotos?**

Isso decide se vale a pena treinar com `ref_img` (identidade) e `latent` (alvo) vindos de **fotos diferentes** da mesma pessoa, em vez da foto idêntica atual (que vaza expressão via CLIP e trava o `s_attr`).

Requer os arquivos originais do CelebA em `./CelebA_data/celeba/`:
- `identity_CelebA.txt`  (filename -> identity_id, sem header)
- `list_attr_celeba.txt` ou `.csv`  (filename -> 40 atributos binários)

Rodar a partir da raiz do projeto (mesma convenção dos outros scripts).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT   = "./CelebA_data/celeba"
IDENTITY_TXT = os.path.join(DATA_ROOT, "identity_CelebA.txt")
ATTR_TXT    = os.path.join(DATA_ROOT, "list_attr_celeba.txt")
ATTR_CSV    = os.path.join(DATA_ROOT, "list_attr_celeba.csv")
IMAGE_DIR   = os.path.join(DATA_ROOT, "img_align_celeba", "img_align_celeba")

for p in [IDENTITY_TXT, ATTR_TXT, ATTR_CSV, IMAGE_DIR]:
    print(f"{'OK ' if os.path.exists(p) else 'FALTA '} {p}")

## 1. Carregar identidades

In [ ]:
if not os.path.exists(IDENTITY_TXT):
    raise FileNotFoundError(
        f"{IDENTITY_TXT} não encontrado. Baixe identity_CelebA.txt "
        "(parte da distribuição oficial do CelebA) e coloque em ./CelebA_data/celeba/."
    )

identity_df = pd.read_csv(
    IDENTITY_TXT,
    sep=r"\s+",
    header=None,
    names=["filename", "identity_id"],
)

print(f"Total de linhas: {len(identity_df)}")
print(f"Identidades únicas: {identity_df['identity_id'].nunique()}")
identity_df.head()

## 2. Carregar atributos (mesma lógica de `utils/utils_celeba.py` / `cache_generator.py`)

In [ ]:
def load_attributes(attr_path):
    if attr_path.endswith(".txt"):
        with open(attr_path, "r") as f:
            lines = f.readlines()
        attr_names = lines[1].split()
        rows = []
        for line in lines[2:]:
            split = line.strip().split()
            filename = split[0]
            attrs = [(int(x) + 1) // 2 for x in split[1:]]  # {-1,1} -> {0,1}
            rows.append([filename] + attrs)
        return pd.DataFrame(rows, columns=["filename"] + attr_names)
    else:
        df = pd.read_csv(attr_path)
        df.columns = ["filename"] + list(df.columns[1:])
        for c in df.columns[1:]:
            df[c] = (df[c] == 1).astype(int)
        return df

attr_path = ATTR_TXT if os.path.exists(ATTR_TXT) else ATTR_CSV
if not os.path.exists(attr_path):
    raise FileNotFoundError(
        "Nenhum arquivo de atributos encontrado (list_attr_celeba.txt/.csv) em "
        f"{DATA_ROOT}."
    )

attr_df = load_attributes(attr_path)
ATTR_NAMES = list(attr_df.columns[1:])
print(f"Amostras: {len(attr_df)} | Atributos: {len(ATTR_NAMES)}")
attr_df.head()

## 3. Merge identidade + atributos

In [ ]:
df = identity_df.merge(attr_df, on="filename", how="inner")
print(f"Amostras após merge: {len(df)} (esperado == total de imagens com atributo E identidade)")
df.head()

## 4. Quantas fotos por identidade?

In [ ]:
photos_per_id = df.groupby("identity_id").size()

print(f"Identidades totais:         {len(photos_per_id)}")
print(f"Média de fotos/identidade:  {photos_per_id.mean():.2f}")
print(f"Mediana:                    {photos_per_id.median():.0f}")
print(f"Mín / Máx:                  {photos_per_id.min()} / {photos_per_id.max()}")
print()
for k in [1, 2, 3, 5, 10]:
    n = (photos_per_id >= k).sum()
    print(f"Identidades com >= {k:>2} fotos: {n:>6} ({100*n/len(photos_per_id):.1f}%)")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(photos_per_id, bins=range(1, photos_per_id.max() + 2), align="left")
plt.xlabel("Fotos por identidade")
plt.ylabel("Número de identidades")
plt.title("Distribuição de fotos por identidade")
plt.yscale("log")
plt.tight_layout()
plt.show()

## 5. O que importa de verdade: quantas identidades têm o atributo *variando* entre fotos?

Ter 2+ fotos não basta — se a pessoa está sorrindo em todas, não ajuda a quebrar o vazamento do CLIP.
Precisamos de identidades onde `Smiling` (e outros atributos) muda de valor entre as fotos disponíveis.

In [ ]:
multi = df[df["identity_id"].isin(photos_per_id[photos_per_id >= 2].index)]
print(f"Amostras em identidades com >=2 fotos: {len(multi)} "
      f"({multi['identity_id'].nunique()} identidades)")

# std > 0 numa coluna binária == o atributo aparece como 0 E como 1 no grupo
varies = multi.groupby("identity_id")[ATTR_NAMES].std() > 0

variety_pct = varies.mean().sort_values(ascending=False) * 100

plt.figure(figsize=(10, 10))
variety_pct.plot(kind="barh")
plt.xlabel("% de identidades (com >=2 fotos) em que o atributo varia")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(variety_pct)

In [ ]:
TARGET_ATTR = "Smiling"

n_multi_id = varies.shape[0]
n_varies_target = varies[TARGET_ATTR].sum()

print(f"Identidades com >=2 fotos:                          {n_multi_id}")
print(f"Dessas, com variação em '{TARGET_ATTR}':                {n_varies_target} "
      f"({100*n_varies_target/n_multi_id:.1f}%)")
print(f"Fração do dataset total de identidades:              "
      f"{100*n_varies_target/len(photos_per_id):.1f}%")

## 6. Amostrar algumas identidades com variação de `Smiling` (checagem visual)

In [ ]:
from PIL import Image

eligible_ids = varies[varies[TARGET_ATTR]].index.tolist()
sample_ids = np.random.RandomState(0).choice(eligible_ids, size=min(4, len(eligible_ids)), replace=False)

fig, axes = plt.subplots(len(sample_ids), 4, figsize=(12, 3 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes[None, :]

for row, ident in enumerate(sample_ids):
    rows = multi[multi["identity_id"] == ident].reset_index(drop=True)
    for col in range(axes.shape[1]):
        ax = axes[row, col]
        ax.axis("off")
        if col >= len(rows):
            continue
        r = rows.iloc[col]
        img_path = os.path.join(IMAGE_DIR, r["filename"])
        if os.path.exists(img_path):
            ax.imshow(Image.open(img_path))
        ax.set_title(f"id={ident} smiling={r[TARGET_ATTR]}", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Exportar tabela de pares candidatos

Para cada identidade com >=2 fotos, salva a lista de arquivos — para o dataset de treino
sortear `ref_img` != `target_img` dentro do mesmo grupo. Identidades com 1 foto só ficam
de fora (caem de volta no comportamento atual: ref == target).

In [ ]:
groups = (
    multi.groupby("identity_id")["filename"]
    .apply(list)
    .reset_index()
    .rename(columns={"filename": "filenames"})
)

out_path = "./CelebA_data/celeba/identity_groups_multi.csv"
groups["filenames"] = groups["filenames"].apply(lambda fs: "|".join(fs))
groups.to_csv(out_path, index=False)
print(f"Salvo: {out_path}  ({len(groups)} identidades, >=2 fotos cada)")